# SynBioCrow 2.4.16 — Dense Generator-Native Evidence Ranking
Upload only the 2.4.16 SOURCE ZIP. This stage does not generate routes. It combines generator-native feasibility/precedent/template metadata with persisted 2.4.15 Rhea/UniProt context and reports score diversity on the development split only.


In [ ]:
from google.colab import drive,files
from pathlib import Path
import csv,hashlib,json,os,shutil,subprocess,sys,time,zipfile

drive.mount("/content/drive",force_remount=False)
T0=time.time()
REPO=Path("/content/SynBioCrow_2_4_16")
SRC=Path("/content/src2416")
RUNOUT=Path("/content/synbiocrow_2_4_16_outputs")
BASE=Path("/content/drive/MyDrive/SynBioCrow/2.4")
OUT=BASE/"native_discrimination_2_4_16"
OUT.mkdir(parents=True,exist_ok=True)

def st(p,m):
    print(f"[2.4.16] {p:5.1f}% | elapsed {(time.time()-T0)/60:5.1f} min | {m}",flush=True)

def run(cmd,cwd=None,timeout=1800):
    print("$"," ".join(map(str,cmd)),flush=True)
    env=os.environ.copy()
    if cwd:
        env["PYTHONPATH"]=str(cwd)+os.pathsep+env.get("PYTHONPATH","")
    q=subprocess.run(list(map(str,cmd)),cwd=cwd,env=env,text=True,
                     stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(q.stdout,flush=True)
    if q.returncode:
        raise subprocess.CalledProcessError(q.returncode,cmd,output=q.stdout)
    return q

st(3,"upload current 2.4.16 SOURCE ZIP only")
u=files.upload()
if len(u)!=1: raise RuntimeError("Upload exactly one 2.4.16 SOURCE ZIP")
name=next(iter(u))
if not name.lower().endswith(".zip"): raise RuntimeError("SOURCE must be a ZIP")
z=Path("/content")/name; z.write_bytes(u[name])
src_sha=hashlib.sha256(z.read_bytes()).hexdigest()
print("SOURCE",name,"sha256",src_sha,flush=True)

st(8,"clone develop/2.4")
shutil.rmtree(REPO,ignore_errors=True)
run(["git","clone","--depth","1","--branch","develop/2.4",
     "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)

st(14,"overlay 2.4.16 source delta")
shutil.rmtree(SRC,ignore_errors=True); SRC.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(z) as q: q.extractall(SRC)
for p in SRC.rglob("*"):
    if p.is_file() and p.name not in {"README_SOURCE.txt","SOURCE_MANIFEST.json"}:
        d=REPO/p.relative_to(SRC)
        d.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(p,d)

st(20,"install lightweight SynBioCrow core")
run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=900)

man=json.loads((REPO/"benchmarks/2.4/split_manifest_2_4_11.json").read_text())
dev=[r["record_id"] for r in man["records"] if r["split"]=="development"]
sealed=[r["record_id"] for r in man["records"] if r["split"]!="development"]

st(28,"bounded discovery of complete development artifacts")
dirs=sorted([p for p in BASE.glob("development_routes_*") if p.is_dir()],
            key=lambda p:p.stat().st_mtime,reverse=True)
chosen=next((d for d in dirs[:20] if all((d/f"{rid}.json").is_file() for rid in dev)),None)
if chosen is None:
    raise FileNotFoundError("No complete development_routes_* directory found")
print("INPUT",chosen,flush=True)
print("SEALED / NOT OPENED",sealed,flush=True)

enriched=BASE/"evidence_enrichment_2_4_15"/"enriched_candidate_rows.json"
if enriched.is_file():
    print("2.4.15 ENRICHMENT",enriched,flush=True)
else:
    enriched=None
    print("2.4.15 enrichment not found; native-only ranking will be used",flush=True)

st(40,"run dense native-evidence discrimination")
shutil.rmtree(RUNOUT,ignore_errors=True); RUNOUT.mkdir(parents=True,exist_ok=True)
cmd=[sys.executable,str(REPO/"scripts/v24_rank_native_evidence_2_4_16.py"),
     "--input-dir",str(chosen),
     "--split-manifest",str(REPO/"benchmarks/2.4/split_manifest_2_4_11.json"),
     "--out-dir",str(RUNOUT)]
if enriched is not None:
    cmd += ["--enriched-rows",str(enriched)]
run(cmd,cwd=REPO,timeout=1800)

st(70,"verify scientific firewall and diversity")
summary=json.loads((RUNOUT/"native_discrimination_summary.json").read_text())
assert summary["generation_invoked"] is False
assert summary["validation_truth_accessed"] is False
assert summary["evaluation_truth_accessed"] is False
assert summary["tuning_performed"] is False
assert summary["labels_used"] is False
assert set(summary["development_record_ids"])==set(dev)
assert set(summary["sealed_nondevelopment_record_ids"])==set(sealed)
print("candidate_count",summary["candidate_count"],flush=True)
print("score_diversity",json.dumps(summary["score_diversity"],indent=2),flush=True)

# Compare against 2.4.14 if its CSV is still present.
old_csv=BASE/"discrimination_2_4_14"/"ranked_routes.csv"
comparison={}
if old_csv.is_file():
    by={}
    with old_csv.open(newline="",encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            rid=row["record_id"]
            try: score=round(float(row["coverage_adjusted_score"]),12)
            except Exception: continue
            by.setdefault(rid,set()).add(score)
    comparison["v2_4_14_distinct_scores"]={k:len(v) for k,v in by.items()}
comparison["v2_4_16_distinct_scores"]={k:v["distinct_score_count"] for k,v in summary["score_diversity"].items()}
(RUNOUT/"score_diversity_comparison.json").write_text(json.dumps(comparison,indent=2,sort_keys=True)+"\n")
print("DIVERSITY COMPARISON",json.dumps(comparison,indent=2),flush=True)

st(86,"persist outputs to Drive")
for p in RUNOUT.iterdir():
    if p.is_file(): shutil.copy2(p,OUT/p.name)
run_manifest={
    "version":"2.4.16","source_zip":name,"source_sha256":src_sha,
    "input_directory":str(chosen),
    "external_enrichment":str(enriched) if enriched else None,
    "generation_invoked":False,"validation_truth_accessed":False,
    "evaluation_truth_accessed":False,"tuning_performed":False,"labels_used":False,
}
(OUT/"run_manifest.json").write_text(json.dumps(run_manifest,indent=2,sort_keys=True)+"\n")

bundle=OUT/"SynBioCrow_2_4_16_DENSE_NATIVE_EVIDENCE_RANKING_ALL_OUTPUTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as q:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p!=bundle: q.write(p,p.name)

st(100,"PASS")
print("BUNDLE",bundle)
print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))


## Rescue download


In [ ]:
from google.colab import files
from pathlib import Path
p=Path("/content/drive/MyDrive/SynBioCrow/2.4/native_discrimination_2_4_16/SynBioCrow_2_4_16_DENSE_NATIVE_EVIDENCE_RANKING_ALL_OUTPUTS.zip")
assert p.is_file(),p
files.download(str(p))
